In [15]:
# Cell 1: Install packages
!pip install flask pandas requests scikit-learn > /dev/null 2>&1
print("🎬 Movie Magic Loading...")

🎬 Movie Magic Loading...


In [16]:
# Cell 2: Enhanced Imports & Setup
import flask
from flask import Flask, render_template_string, request, jsonify
import requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import json
import threading
from collections import defaultdict
import random
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity
import time

app = Flask(__name__)

# TMDB API configuration
API_KEY = "a86e20867eb38f2c0805af03db487033"
BASE_URL = "https://api.themoviedb.org/3"
POSTER_BASE_URL = "https://image.tmdb.org/t/p/w500"

# Enhanced Genre mappings with emojis
GENRES = {
    28: "🎯 Action", 12: "🏔️ Adventure", 16: "🐰 Animation", 35: "😂 Comedy",
    80: "🔫 Crime", 18: "🎭 Drama", 10751: "👨‍👩‍👧‍👦 Family", 14: "🧙 Fantasy",
    36: "📜 History", 27: "👻 Horror", 10402: "🎵 Music", 9648: "🕵️ Mystery",
    10749: "💖 Romance", 878: "🚀 Sci-Fi", 53: "😱 Thriller", 10752: "⚔️ War",
    37: "🤠 Western", 99: "📽️ Documentary", 10770: "📺 TV Movie"
}

# Enhanced Countries with flags
COUNTRIES = {
    'US': '🇺🇸 United States', 'IN': '🇮🇳 India', 'GB': '🇬🇧 United Kingdom',
    'CA': '🇨🇦 Canada', 'AU': '🇦🇺 Australia', 'DE': '🇩🇪 Germany',
    'FR': '🇫🇷 France', 'JP': '🇯🇵 Japan', 'KR': '🇰🇷 South Korea',
    'CN': '🇨🇳 China', 'BR': '🇧🇷 Brazil', 'IT': '🇮🇹 Italy',
    'ES': '🇪🇸 Spain', 'RU': '🇷🇺 Russia', 'MX': '🇲🇽 Mexico'
}

LANGUAGES = {
    'en': '🇺🇸 English', 'hi': '🇮🇳 Hindi', 'ta': '🇮🇳 Tamil', 'te': '🇮🇳 Telugu',
    'ml': '🇮🇳 Malayalam', 'kn': '🇮🇳 Kannada', 'bn': '🇮🇳 Bengali', 'mr': '🇮🇳 Marathi',
    'gu': '🇮🇳 Gujarati', 'pa': '🇮🇳 Punjabi', 'es': '🇪🇸 Spanish', 'fr': '🇫🇷 French',
    'de': '🇩🇪 German', 'it': '🇮🇹 Italian', 'ja': '🇯🇵 Japanese', 'ko': '🇰🇷 Korean',
    'zh': '🇨🇳 Chinese', 'ru': '🇷🇺 Russian', 'pt': '🇵🇹 Portuguese'
}

INDIAN_LANGUAGES = {k: v for k, v in LANGUAGES.items() if k in ['hi', 'ta', 'te', 'ml', 'kn', 'bn', 'mr', 'gu', 'pa']}

# Enhanced Kids Animations
KIDS_BACKGROUND_ANIMATIONS = [
    "https://assets1.lottiefiles.com/packages/lf20_2cw3jkys.json",   # Cinema
    "https://assets1.lottiefiles.com/packages/lf20_uknSzR.json",     # Movie camera
    "https://assets1.lottiefiles.com/packages/lf20_isdnS2.json",     # Popcorn
    "https://assets1.lottiefiles.com/packages/lf20_t9pGcL.json",     # Clapperboard
    "https://assets1.lottiefiles.com/packages/lf20_5itp2rli.json",   # Film reel
    "https://assets1.lottiefiles.com/packages/lf20_obdsfe9r.json",   # Movie tickets
]

# AI storage
movie_features = {}

In [17]:
# Cell 3: Enhanced AI System
class AdvancedMovieAI:
    def __init__(self):
        self.movie_similarity = {}
        self.user_preferences = defaultdict(dict)
        self.recommendation_history = defaultdict(list)

    def build_similarity_model(self, movies):
        """Advanced content-based similarity with multiple factors"""
        print("🧠 Building Advanced AI Model...")
        for movie in movies:
            if movie['id'] not in self.movie_similarity:
                self.movie_similarity[movie['id']] = {
                    'genres': set(movie.get('genre_ids', [])),
                    'year': int(movie.get('release_date', '2000')[:4]) if movie.get('release_date') else 2000,
                    'rating': movie.get('vote_average', 0),
                    'popularity': movie.get('popularity', 0),
                    'vote_count': movie.get('vote_count', 0),
                    'language': movie.get('original_language', 'en')
                }
        print(f"✅ AI Model Trained on {len(self.movie_similarity)} movies")

    def get_similar_movies(self, movie_id, all_movies, top_n=12):
        """Get similar movies with advanced scoring"""
        if movie_id not in self.movie_similarity:
            return self._get_fallback_recommendations(all_movies, top_n)

        target = self.movie_similarity[movie_id]
        similarities = []

        for movie in all_movies:
            if movie['id'] == movie_id or movie['id'] not in self.movie_similarity:
                continue

            other = self.movie_similarity[movie['id']]

            # Advanced similarity calculation
            genre_similarity = len(target['genres'].intersection(other['genres']))
            year_similarity = 1 - abs(target['year'] - other['year']) / 50
            rating_similarity = 1 - abs(target['rating'] - other['rating']) / 10
            popularity_similarity = min(target['popularity'], other['popularity']) / max(target['popularity'], other['popularity'], 1)
            language_similarity = 1 if target['language'] == other['language'] else 0.3

            # Weighted total similarity
            total_similarity = (
                genre_similarity * 0.35 +
                rating_similarity * 0.25 +
                year_similarity * 0.15 +
                popularity_similarity * 0.15 +
                language_similarity * 0.10
            )

            similarities.append((movie, total_similarity))

        similarities.sort(key=lambda x: x[1], reverse=True)
        recommendations = [movie for movie, _ in similarities[:top_n]]

        # Store recommendation history
        self.recommendation_history[movie_id].extend([m['id'] for m in recommendations])

        return recommendations if recommendations else self._get_fallback_recommendations(all_movies, top_n)

    def _get_fallback_recommendations(self, all_movies, top_n):
        """Fallback to trending movies if no similar ones found"""
        scored_movies = []
        for movie in all_movies:
            score = (
                movie.get('popularity', 0) * 0.4 +
                movie.get('vote_average', 0) * 0.3 +
                (1 if movie.get('release_date', '').startswith('202') else 0.5) * 0.3
            )
            scored_movies.append((movie, score))

        scored_movies.sort(key=lambda x: x[1], reverse=True)
        return [movie for movie, _ in scored_movies[:top_n]]

    def get_personalized_recommendations(self, user_id, all_movies, top_n=15):
        """Get personalized recommendations based on user history"""
        user_history = self.recommendation_history.get(user_id, [])
        if not user_history:
            return self._get_fallback_recommendations(all_movies, top_n)

        # Analyze user preferences from history
        preferred_genres = defaultdict(int)
        for movie_id in user_history[-10:]:  # Last 10 interactions
            if movie_id in self.movie_similarity:
                for genre in self.movie_similarity[movie_id]['genres']:
                    preferred_genres[genre] += 1

        # Score movies based on user preferences
        scored_movies = []
        for movie in all_movies:
            if movie['id'] in user_history:
                continue

            score = 0
            if movie['id'] in self.movie_similarity:
                movie_data = self.movie_similarity[movie['id']]
                # Genre preference score
                for genre in movie_data['genres']:
                    score += preferred_genres.get(genre, 0) * 2
                # Recency bonus
                if movie_data['year'] >= 2020:
                    score += 5
                # Rating bonus
                score += movie_data['rating'] * 0.5

            scored_movies.append((movie, score))

        scored_movies.sort(key=lambda x: x[1], reverse=True)
        return [movie for movie, _ in scored_movies[:top_n]]

# Initialize Advanced AI
movie_ai = AdvancedMovieAI()

In [18]:
# Cell 4: Enhanced API Functions
def fetch_recent_movies(pages=2):
    """Fetch recently released movies with enhanced data"""
    recent_movies = []
    current_date = datetime.now()
    one_year_ago = current_date - timedelta(days=365)

    for page in range(1, pages + 1):
        url = f"{BASE_URL}/discover/movie"
        params = {
            'api_key': API_KEY,
            'page': page,
            'language': 'en-US',
            'sort_by': 'popularity.desc',
            'primary_release_date.gte': one_year_ago.strftime('%Y-%m-%d'),
            'primary_release_date.lte': current_date.strftime('%Y-%m-%d')
        }
        try:
            response = requests.get(url, params=params, timeout=10)
            if response.status_code == 200:
                recent_movies.extend(response.json().get('results', []))
        except Exception as e:
            print(f"⚠️ Error fetching recent movies: {e}")
    return recent_movies

def fetch_popular_movies(pages=3):
    """Fetch popular movies with enhanced data"""
    all_movies = []
    for page in range(1, pages + 1):
        url = f"{BASE_URL}/movie/popular"
        params = {
            'api_key': API_KEY,
            'page': page,
            'language': 'en-US'
        }
        try:
            response = requests.get(url, params=params, timeout=10)
            if response.status_code == 200:
                all_movies.extend(response.json().get('results', []))
        except Exception as e:
            print(f"⚠️ Error fetching popular movies: {e}")
    return all_movies

def fetch_kids_movies_enhanced(pages=3):
    """Fetch kids movies with better filtering"""
    kids_movies = []
    for page in range(1, pages + 1):
        url = f"{BASE_URL}/discover/movie"
        params = {
            'api_key': API_KEY,
            'with_genres': '16,10751,35',  # Animation + Family + Comedy
            'certification_country': 'US',
            'certification.lte': 'PG',
            'page': page,
            'language': 'en-US',
            'sort_by': 'popularity.desc',
            'primary_release_date.gte': '2000-01-01'
        }
        try:
            response = requests.get(url, params=params, timeout=10)
            if response.status_code == 200:
                kids_movies.extend(response.json().get('results', []))
        except Exception as e:
            print(f"⚠️ Error fetching kids movies: {e}")
    return kids_movies

def search_movies_enhanced(query, page=1):
    """Enhanced movie search"""
    url = f"{BASE_URL}/search/movie"
    params = {
        'api_key': API_KEY,
        'query': query,
        'page': page,
        'language': 'en-US'
    }
    try:
        response = requests.get(url, params=params, timeout=10)
        if response.status_code == 200:
            return response.json().get('results', [])
    except Exception as e:
        print(f"⚠️ Error searching movies: {e}")
    return []

def fetch_movies_with_filters(genre=None, language=None, year_from=1980, year_to=2025, page=1):
    """Fetch movies with multiple filters"""
    url = f"{BASE_URL}/discover/movie"
    params = {
        'api_key': API_KEY,
        'page': page,
        'language': 'en-US',
        'sort_by': 'popularity.desc',
        'primary_release_date.gte': f'{year_from}-01-01',
        'primary_release_date.lte': f'{year_to}-12-31'
    }

    if genre:
        params['with_genres'] = genre
    if language:
        params['with_original_language'] = language

    try:
        response = requests.get(url, params=params, timeout=10)
        if response.status_code == 200:
            return response.json().get('results', [])
    except Exception as e:
        print(f"⚠️ Error fetching filtered movies: {e}")
    return []

def fetch_movie_details_enhanced(movie_id):
    """Fetch detailed movie information"""
    url = f"{BASE_URL}/movie/{movie_id}"
    params = {
        'api_key': API_KEY,
        'language': 'en-US',
        'append_to_response': 'videos,credits,similar'
    }
    try:
        response = requests.get(url, params=params, timeout=10)
        if response.status_code == 200:
            return response.json()
    except Exception as e:
        print(f"⚠️ Error fetching movie details: {e}")
    return None

def get_youtube_trailer(movie_data):
    """Get YouTube trailer URL"""
    try:
        videos = movie_data.get('videos', {}).get('results', [])
        trailer_videos = [v for v in videos if v['type'] == 'Trailer' and v['site'] == 'YouTube']

        if trailer_videos:
            official_trailers = [v for v in trailer_videos if 'official' in v['name'].lower()]
            trailer_to_show = official_trailers[0] if official_trailers else trailer_videos[0]
            return f"https://www.youtube.com/watch?v={trailer_to_show['key']}"
    except Exception as e:
        print(f"⚠️ Error getting trailer: {e}")
    return None

In [19]:
# Cell 5: Clean HTML Template (No AI, Only Similar Movies)
HTML_TEMPLATE = """
<!DOCTYPE html>
<html>
<head>
    <title>🎬 Ultimate Movie Finder</title>
    <meta charset="utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <script src="https://unpkg.com/@lottiefiles/lottie-player@latest/dist/lottie-player.js"></script>
    <link href="https://fonts.googleapis.com/css2?family=Poppins:wght@300;400;500;600;700&display=swap" rel="stylesheet">
    <style>
        * {
            margin: 0;
            padding: 0;
            box-sizing: border-box;
            font-family: 'Poppins', sans-serif;
        }

        body {
            background: linear-gradient(135deg, #0f0c29, #302b63, #24243e);
            color: white;
            min-height: 100vh;
            transition: all 0.5s ease;
            position: relative;
            overflow-x: hidden;
        }

        /* Regular Mode Sparkles */
        .sparkle {
            position: absolute;
            width: 4px;
            height: 4px;
            background: #FFD700;
            border-radius: 50%;
            animation: sparkle 3s infinite;
            pointer-events: none;
            z-index: 1;
        }

        @keyframes sparkle {
            0%, 100% { opacity: 0; transform: scale(0); }
            50% { opacity: 1; transform: scale(1); }
        }

        /* Kids Mode Big Sparkles */
        .big-sparkle {
            position: absolute;
            width: 8px;
            height: 8px;
            background: radial-gradient(circle, #FFD700, #FF6B6B, #4ECDC4);
            border-radius: 50%;
            animation: bigSparkle 2s infinite;
            pointer-events: none;
            z-index: 1;
            box-shadow: 0 0 20px #FFD700;
        }

        @keyframes bigSparkle {
            0%, 100% { opacity: 0; transform: scale(0) rotate(0deg); }
            50% { opacity: 1; transform: scale(1.5) rotate(180deg); }
        }

        /* Kids Background Animations */
        .kids-bg-animation {
            position: fixed;
            top: 0;
            left: 0;
            width: 100%;
            height: 100%;
            pointer-events: none;
            z-index: 0;
            opacity: 0.1;
        }

        .floating-animation {
            position: absolute;
            animation: floatAround 20s infinite linear;
        }

        @keyframes floatAround {
            0% { transform: translate(0, 0) rotate(0deg); }
            25% { transform: translate(100px, 100px) rotate(90deg); }
            50% { transform: translate(200px, 0) rotate(180deg); }
            75% { transform: translate(100px, -100px) rotate(270deg); }
            100% { transform: translate(0, 0) rotate(360deg); }
        }

        .container {
            max-width: 1400px;
            margin: 0 auto;
            padding: 20px;
            position: relative;
            z-index: 2;
        }

        .header {
            text-align: center;
            padding: 40px 0;
            background: linear-gradient(135deg, rgba(255,255,255,0.1), rgba(255,255,255,0.05));
            border-radius: 25px;
            margin-bottom: 30px;
            backdrop-filter: blur(20px);
            border: 1px solid rgba(255,255,255,0.1);
            position: relative;
            overflow: hidden;
        }

        .header::before {
            content: '';
            position: absolute;
            top: -50%;
            left: -50%;
            width: 200%;
            height: 200%;
            background: linear-gradient(45deg, transparent, rgba(255,255,255,0.1), transparent);
            animation: shine 3s infinite;
        }

        @keyframes shine {
            0% { transform: translateX(-100%) translateY(-100%) rotate(45deg); }
            100% { transform: translateX(100%) translateY(100%) rotate(45deg); }
        }

        .header h1 {
            font-size: 3.5rem;
            margin-bottom: 15px;
            background: linear-gradient(45deg, #FFD700, #FF6B6B, #4ECDC4, #667eea);
            -webkit-background-clip: text;
            -webkit-text-fill-color: transparent;
            background-size: 300% 300%;
            animation: gradientShift 3s ease infinite;
            font-weight: 700;
            text-shadow: 0 0 30px rgba(255,215,0,0.3);
        }

        @keyframes gradientShift {
            0% { background-position: 0% 50%; }
            50% { background-position: 100% 50%; }
            100% { background-position: 0% 50%; }
        }

        .nav-bar {
            display: flex;
            justify-content: space-between;
            align-items: center;
            margin-bottom: 30px;
            background: rgba(255,255,255,0.1);
            padding: 15px 25px;
            border-radius: 15px;
            backdrop-filter: blur(15px);
            border: 1px solid rgba(255,255,255,0.1);
        }

        .home-btn {
            background: linear-gradient(45deg, #667eea, #764ba2);
            color: white;
            border: none;
            padding: 12px 25px;
            border-radius: 25px;
            cursor: pointer;
            font-weight: 600;
            transition: all 0.3s ease;
            text-decoration: none;
            display: flex;
            align-items: center;
            gap: 8px;
        }

        .home-btn:hover {
            transform: translateY(-2px);
            box-shadow: 0 5px 15px rgba(102, 126, 234, 0.4);
        }

        .mode-selector {
            display: flex;
            gap: 15px;
        }

        .mode-btn {
            padding: 12px 25px;
            border: none;
            border-radius: 25px;
            font-size: 1rem;
            font-weight: 600;
            cursor: pointer;
            transition: all 0.3s ease;
            background: rgba(255,255,255,0.1);
            color: white;
            border: 1px solid rgba(255,255,255,0.2);
            position: relative;
            overflow: hidden;
        }

        .mode-btn::before {
            content: '';
            position: absolute;
            top: 0;
            left: -100%;
            width: 100%;
            height: 100%;
            background: linear-gradient(90deg, transparent, rgba(255,255,255,0.2), transparent);
            transition: 0.5s;
        }

        .mode-btn:hover::before {
            left: 100%;
        }

        .mode-btn.active {
            background: linear-gradient(45deg, #FFD700, #FF6B6B);
            transform: scale(1.05);
            box-shadow: 0 5px 15px rgba(255, 107, 107, 0.3);
        }

        .mode-btn:hover {
            transform: translateY(-2px);
            box-shadow: 0 5px 15px rgba(255,255,255,0.2);
        }

        .filters-container {
            background: rgba(255, 255, 255, 0.15);
            padding: 30px;
            border-radius: 20px;
            margin: 20px 0;
            backdrop-filter: blur(15px);
            border: 1px solid rgba(255, 255, 255, 0.2);
        }

        .filter-row {
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(200px, 1fr));
            gap: 20px;
            margin-bottom: 20px;
        }

        .filter-group {
            display: flex;
            flex-direction: column;
        }

        .filter-label {
            font-weight: 600;
            margin-bottom: 10px;
            color: #FFD700;
            font-size: 0.9rem;
            text-transform: uppercase;
            letter-spacing: 1px;
            text-shadow: 1px 1px 2px rgba(0,0,0,0.5);
        }

        .filter-select, .search-box {
            width: 100%;
            padding: 12px 15px;
            border: none;
            border-radius: 12px;
            background: rgba(255, 255, 255, 0.15);
            color: white;
            font-size: 1rem;
            transition: all 0.3s ease;
            border: 1px solid rgba(255,255,255,0.3);
        }

        .filter-select option {
            background: #2a2a2a;
            color: white;
        }

        .filter-select:focus, .search-box:focus {
            outline: none;
            box-shadow: 0 0 20px rgba(255, 215, 0, 0.3);
            border-color: #FFD700;
            background: rgba(255,255,255,0.2);
        }

        .search-box::placeholder {
            color: rgba(255,255,255,0.7);
        }

        .year-range {
            display: flex;
            gap: 10px;
            align-items: center;
        }

        .year-input {
            width: 80px;
            padding: 10px;
            border: none;
            border-radius: 8px;
            text-align: center;
            background: rgba(255, 255, 255, 0.15);
            color: white;
            border: 1px solid rgba(255,255,255,0.3);
        }

        .apply-filters {
            background: linear-gradient(45deg, #FF6B6B, #FF8E53);
            color: white;
            border: none;
            padding: 15px 35px;
            border-radius: 25px;
            cursor: pointer;
            font-size: 1.1rem;
            font-weight: 600;
            margin-top: 15px;
            transition: all 0.3s ease;
            box-shadow: 0 4px 15px rgba(255, 107, 107, 0.3);
            position: relative;
            overflow: hidden;
        }

        .apply-filters:hover {
            transform: translateY(-3px) scale(1.02);
            box-shadow: 0 8px 25px rgba(255, 107, 107, 0.4);
        }

        .kids-welcome {
            text-align: center;
            margin: 30px 0;
            padding: 40px;
            background: linear-gradient(135deg, rgba(255,215,0,0.2), rgba(255,107,107,0.2));
            border-radius: 25px;
            backdrop-filter: blur(15px);
            border: 2px solid rgba(255,255,255,0.2);
            display: none;
        }

        .kids-welcome h2 {
            color: #FFD700;
            font-size: 2.5rem;
            margin-bottom: 15px;
            text-shadow: 0 0 10px rgba(255,215,0,0.5);
        }

        .kids-welcome p {
            color: white;
            font-size: 1.2rem;
            opacity: 0.9;
        }

        .movie-grid {
            display: grid;
            grid-template-columns: repeat(auto-fill, minmax(280px, 1fr));
            gap: 30px;
            margin: 40px 0;
        }

        .movie-card {
            background: linear-gradient(135deg, rgba(255,255,255,0.15), rgba(255,255,255,0.08));
            border-radius: 20px;
            padding: 20px;
            text-align: center;
            transition: all 0.4s ease;
            backdrop-filter: blur(15px);
            border: 1px solid rgba(255,255,255,0.15);
            position: relative;
            overflow: hidden;
        }

        .movie-card::before {
            content: '';
            position: absolute;
            top: 0;
            left: 0;
            right: 0;
            height: 4px;
            background: linear-gradient(45deg, #FFD700, #FF6B6B, #4ECDC4);
        }

        .movie-card:hover {
            transform: translateY(-10px) scale(1.02);
            box-shadow: 0 20px 40px rgba(0,0,0,0.3);
            border-color: rgba(255,215,0,0.3);
        }

        .kids-movie-card {
            background: linear-gradient(135deg, rgba(78, 205, 196, 0.2), rgba(255, 107, 107, 0.2));
            border: 2px solid rgba(255, 215, 0, 0.3);
            animation: kidCardPulse 2s infinite alternate;
        }

        @keyframes kidCardPulse {
            0% { box-shadow: 0 10px 30px rgba(78, 205, 196, 0.3); }
            100% { box-shadow: 0 15px 40px rgba(255, 215, 0, 0.5); }
        }

        .movie-poster {
            width: 100%;
            height: 350px;
            object-fit: cover;
            border-radius: 15px;
            margin-bottom: 15px;
            border: 2px solid rgba(255,255,255,0.2);
            transition: all 0.3s ease;
        }

        .movie-card:hover .movie-poster {
            border-color: #FFD700;
            transform: scale(1.05);
        }

        .movie-title {
            font-size: 1.1rem;
            font-weight: 600;
            margin: 10px 0;
            color: white;
            line-height: 1.4;
            min-height: 3em;
            display: flex;
            align-items: center;
            justify-content: center;
            text-shadow: 1px 1px 2px rgba(0,0,0,0.5);
        }

        .movie-info {
            font-size: 0.9rem;
            color: #ddd;
            margin-bottom: 10px;
            text-shadow: 1px 1px 2px rgba(0,0,0,0.5);
        }

        .movie-language {
            background: linear-gradient(45deg, rgba(255,215,0,0.3), rgba(255,107,107,0.3));
            padding: 6px 12px;
            border-radius: 15px;
            font-size: 0.8rem;
            margin: 8px 0;
            display: inline-block;
            border: 1px solid rgba(255,215,0,0.4);
            color: white;
            text-shadow: 1px 1px 2px rgba(0,0,0,0.5);
        }

        .kids-theme {
            background: linear-gradient(135deg, #FF6B6B, #FFD700, #4ECDC4, #667eea) !important;
            background-size: 400% 400% !important;
            animation: gradientMove 6s ease infinite !important;
        }

        @keyframes gradientMove {
            0% { background-position: 0% 50%; }
            50% { background-position: 100% 50%; }
            100% { background-position: 0% 50%; }
        }

        .details-btn {
            background: linear-gradient(45deg, #667eea, #764ba2);
            color: white;
            border: none;
            padding: 12px 25px;
            border-radius: 25px;
            cursor: pointer;
            font-weight: 600;
            margin-top: 10px;
            transition: all 0.3s ease;
            width: 100%;
            position: relative;
            overflow: hidden;
        }

        .details-btn:hover {
            transform: translateY(-2px);
            box-shadow: 0 5px 15px rgba(102, 126, 234, 0.4);
        }

        .movie-details {
            background: rgba(255, 255, 255, 0.15);
            padding: 40px;
            border-radius: 25px;
            margin: 20px 0;
            backdrop-filter: blur(20px);
            border: 1px solid rgba(255, 255, 255, 0.2);
            display: none;
        }

        .trailer-container {
            margin: 25px 0;
            text-align: center;
        }

        .trailer-btn {
            background: linear-gradient(45deg, #FF0000, #CC0000);
            color: white;
            border: none;
            padding: 15px 35px;
            border-radius: 25px;
            font-size: 1.2rem;
            font-weight: 600;
            cursor: pointer;
            text-decoration: none;
            display: inline-flex;
            align-items: center;
            gap: 12px;
            transition: all 0.3s ease;
            box-shadow: 0 4px 15px rgba(255, 0, 0, 0.3);
        }

        .trailer-btn:hover {
            transform: scale(1.05);
            box-shadow: 0 6px 20px rgba(255, 0, 0, 0.4);
            text-decoration: none;
            color: white;
        }

        .back-btn {
            background: linear-gradient(45deg, #667eea, #764ba2);
            color: white;
            border: none;
            padding: 12px 25px;
            border-radius: 25px;
            cursor: pointer;
            font-size: 1rem;
            font-weight: 600;
            margin: 20px 0;
            transition: all 0.3s ease;
        }

        .back-btn:hover {
            transform: translateY(-2px);
            box-shadow: 0 5px 15px rgba(102, 126, 234, 0.4);
        }

        .loading {
            text-align: center;
            padding: 50px;
            font-size: 1.3rem;
            color: #FFD700;
        }

        .stats {
            background: rgba(255, 255, 255, 0.15);
            padding: 20px;
            border-radius: 15px;
            margin: 20px 0;
            text-align: center;
            backdrop-filter: blur(15px);
            border: 1px solid rgba(255,255,255,0.2);
            color: white;
            text-shadow: 1px 1px 2px rgba(0,0,0,0.5);
        }

        /* Similar Movies Section */
        .similar-movies {
            margin: 40px 0;
            display: none;
        }

        .similar-movies h3 {
            color: #FFD700;
            font-size: 1.8rem;
            margin-bottom: 25px;
            text-align: center;
            text-shadow: 0 0 10px rgba(255,215,0,0.3);
        }

        .similar-grid {
            display: grid;
            grid-template-columns: repeat(auto-fill, minmax(250px, 1fr));
            gap: 25px;
        }
    </style>
</head>
<body>
    <div class="container">
        <!-- Navigation Bar -->
        <div class="nav-bar">
            <button class="home-btn" onclick="goHome()">
                🏠 Home
            </button>
            <div class="mode-selector">
                <button class="mode-btn active" onclick="setMode('regular')">🎬 Regular Mode</button>
                <button class="mode-btn" onclick="setMode('kids')">🎪 Kids Fun Mode</button>
            </div>
        </div>

        <!-- Header -->
        <div class="header" id="header">
            <h1>🎬 Ultimate Movie Finder</h1>
            <p>Discover Amazing Movies from 1980 to 2025</p>
        </div>

        <!-- Kids Welcome Message -->
        <div class="kids-welcome" id="kidsWelcome">
            <h2>🎪 Welcome to Kids Fun Zone! 🎪</h2>
            <p>Discover magical movies perfect for family time! ✨</p>
        </div>

        <!-- Filters -->
        <div class="filters-container">
            <div class="filter-row">
                <div class="filter-group">
                    <div class="filter-label">🔍 Search Movies</div>
                    <input type="text" class="search-box" id="searchInput" placeholder="Type movie name...">
                </div>

                <div class="filter-group">
                    <div class="filter-label">🎭 Genre</div>
                    <select class="filter-select" id="genreSelect">
                        <option value="">All Genres</option>
                    </select>
                </div>

                <div class="filter-group">
                    <div class="filter-label">🌍 Country</div>
                    <select class="filter-select" id="countrySelect">
                        <option value="">All Countries</option>
                    </select>
                </div>
            </div>

            <div class="filter-row">
                <div class="filter-group">
                    <div class="filter-label">🗣️ Language Type</div>
                    <select class="filter-select" id="languageTypeSelect" onchange="toggleLanguageSelection()">
                        <option value="">All Languages</option>
                        <option value="indian">Indian Languages</option>
                        <option value="international">International</option>
                    </select>
                </div>

                <div class="filter-group">
                    <div class="filter-label">🗣️ Specific Language</div>
                    <select class="filter-select" id="languageSelect">
                        <option value="">All Languages</option>
                    </select>
                </div>

                <div class="filter-group">
                    <div class="filter-label">📅 Release Year</div>
                    <div class="year-range">
                        <input type="number" class="year-input" id="yearFrom" placeholder="1980" min="1980" max="2025" value="1980">
                        <span>to</span>
                        <input type="number" class="year-input" id="yearTo" placeholder="2025" min="1980" max="2025" value="2025">
                    </div>
                </div>
            </div>

            <button class="apply-filters" onclick="applyFilters()">
                🎬 Find Movies
            </button>
        </div>

        <!-- Stats -->
        <div class="stats" id="stats">
            Ready to explore amazing movies! 🎉
        </div>

        <!-- Movies Grid -->
        <div id="movieGrid" class="movie-grid">
            <!-- Movies will be loaded here -->
        </div>

        <!-- Similar Movies Section -->
        <div id="similarMovies" class="similar-movies">
            <h3>🎯 Similar Movies You Might Like</h3>
            <div id="similarGrid" class="similar-grid">
                <!-- Similar movies will be loaded here -->
            </div>
        </div>

        <!-- Movie Details -->
        <div id="movieDetails" class="movie-details">
            <!-- Movie details will be loaded here -->
        </div>
    </div>

    <script>
        let currentMode = 'regular';
        let allMovies = [];
        let currentFilters = {};

        // Kids background animations
        const kidsAnimations = {{ kids_animations | tojson }};

        // Create sparkle effects for regular mode
        function createSparkles() {
            const sparkleCount = 30;
            for (let i = 0; i < sparkleCount; i++) {
                const sparkle = document.createElement('div');
                sparkle.className = 'sparkle';
                sparkle.style.left = Math.random() * 100 + 'vw';
                sparkle.style.top = Math.random() * 100 + 'vh';
                sparkle.style.animationDelay = Math.random() * 3 + 's';
                document.body.appendChild(sparkle);
            }
        }

        // Create big sparkles for kids mode
        function createBigSparkles() {
            const bigSparkleCount = 20;
            for (let i = 0; i < bigSparkleCount; i++) {
                const bigSparkle = document.createElement('div');
                bigSparkle.className = 'big-sparkle';
                bigSparkle.style.left = Math.random() * 100 + 'vw';
                bigSparkle.style.top = Math.random() * 100 + 'vh';
                bigSparkle.style.animationDelay = Math.random() * 2 + 's';
                bigSparkle.style.animationDuration = (Math.random() * 2 + 1) + 's';
                document.body.appendChild(bigSparkle);
            }
        }

        // Create floating animations for kids mode
        function createFloatingAnimations() {
            kidsAnimations.forEach((animation, index) => {
                const floatingAnim = document.createElement('div');
                floatingAnim.className = 'kids-bg-animation floating-animation';
                floatingAnim.innerHTML = `
                    <lottie-player
                        src="${animation}"
                        background="transparent"
                        speed="${0.5 + Math.random() * 0.5}"
                        style="width: ${80 + Math.random() * 120}px; height: ${80 + Math.random() * 120}px;"
                        loop
                        autoplay
                    ></lottie-player>
                `;
                floatingAnim.style.left = Math.random() * 80 + 10 + '%';
                floatingAnim.style.top = Math.random() * 80 + 10 + '%';
                floatingAnim.style.animationDuration = (15 + Math.random() * 20) + 's';
                document.body.appendChild(floatingAnim);
            });
        }

        // Clear all animations
        function clearAnimations() {
            document.querySelectorAll('.sparkle, .big-sparkle, .kids-bg-animation').forEach(el => el.remove());
        }

        // Initialize filters
        function initializeFilters() {
            // Populate genres
            const genreSelect = document.getElementById('genreSelect');
            {% for genre in genres %}
            const option{{ genre.id }} = document.createElement('option');
            option{{ genre.id }}.value = '{{ genre.id }}';
            option{{ genre.id }}.textContent = '{{ genre.name }}';
            genreSelect.appendChild(option{{ genre.id }});
            {% endfor %}

            // Populate countries
            const countrySelect = document.getElementById('countrySelect');
            {% for code, name in countries.items() %}
            const countryOption{{ code }} = document.createElement('option');
            countryOption{{ code }}.value = '{{ code }}';
            countryOption{{ code }}.textContent = '{{ name }}';
            countrySelect.appendChild(countryOption{{ code }});
            {% endfor %}

            // Populate all languages
            const languageSelect = document.getElementById('languageSelect');
            {% for code, name in languages.items() %}
            const langOption{{ code }} = document.createElement('option');
            langOption{{ code }}.value = '{{ code }}';
            langOption{{ code }}.textContent = '{{ name }}';
            languageSelect.appendChild(langOption{{ code }});
            {% endfor %}

            // Create initial sparkles
            createSparkles();
        }

        function toggleLanguageSelection() {
            const languageType = document.getElementById('languageTypeSelect').value;
            const languageSelect = document.getElementById('languageSelect');

            // Clear existing options except "All Languages"
            while (languageSelect.options.length > 1) {
                languageSelect.remove(1);
            }

            if (languageType === 'indian') {
                {% for code, name in indian_languages.items() %}
                const option{{ code }} = document.createElement('option');
                option{{ code }}.value = '{{ code }}';
                option{{ code }}.textContent = '{{ name }}';
                languageSelect.appendChild(option{{ code }});
                {% endfor %}
            } else if (languageType === 'international') {
                {% for code, name in languages.items() %}
                {% if code not in ['hi', 'ta', 'te', 'ml', 'kn', 'bn', 'mr', 'gu', 'pa'] %}
                const option{{ code }} = document.createElement('option');
                option{{ code }}.value = '{{ code }}';
                option{{ code }}.textContent = '{{ name }}';
                languageSelect.appendChild(option{{ code }});
                {% endif %}
                {% endfor %}
            } else {
                {% for code, name in languages.items() %}
                const option{{ code }} = document.createElement('option');
                option{{ code }}.value = '{{ code }}';
                option{{ code }}.textContent = '{{ name }}';
                languageSelect.appendChild(option{{ code }});
                {% endfor %}
            }
        }

        function setMode(mode) {
            currentMode = mode;
            const header = document.getElementById('header');
            const kidsWelcome = document.getElementById('kidsWelcome');
            const modeButtons = document.querySelectorAll('.mode-btn');
            const body = document.body;

            // Clear existing animations
            clearAnimations();

            modeButtons.forEach(btn => btn.classList.remove('active'));
            event.target.classList.add('active');

            if (mode === 'kids') {
                header.innerHTML = '<h1>🎪 Kids Fun Movie Zone 🎪</h1><p>Magical movies for the whole family! ✨</p>';
                kidsWelcome.style.display = 'block';
                body.className = 'kids-theme';

                // Create kids mode animations
                createBigSparkles();
                createFloatingAnimations();

            } else {
                header.innerHTML = '<h1>🎬 Ultimate Movie Finder</h1><p>Discover Amazing Movies from 1980 to 2025</p>';
                kidsWelcome.style.display = 'none';
                body.className = '';

                // Create regular mode sparkles
                createSparkles();
            }

            // Clear movie details and similar movies when switching modes
            document.getElementById('movieDetails').style.display = 'none';
            document.getElementById('similarMovies').style.display = 'none';
            document.getElementById('movieGrid').style.display = 'grid';

            applyFilters();
        }

        function goHome() {
            // Reset to initial state
            document.getElementById('movieDetails').style.display = 'none';
            document.getElementById('similarMovies').style.display = 'none';
            document.getElementById('movieGrid').style.display = 'grid';
            document.getElementById('searchInput').value = '';
            document.getElementById('genreSelect').value = '';
            document.getElementById('countrySelect').value = '';
            document.getElementById('languageTypeSelect').value = '';
            document.getElementById('languageSelect').value = '';
            document.getElementById('yearFrom').value = '1980';
            document.getElementById('yearTo').value = '2025';

            applyFilters();
        }

        function applyFilters() {
            const searchQuery = document.getElementById('searchInput').value;
            const genre = document.getElementById('genreSelect').value;
            const country = document.getElementById('countrySelect').value;
            const language = document.getElementById('languageSelect').value;
            const yearFrom = document.getElementById('yearFrom').value || '1980';
            const yearTo = document.getElementById('yearTo').value || '2025';

            currentFilters = {
                mode: currentMode,
                search: searchQuery,
                genre: genre,
                country: country,
                language: language,
                year_from: yearFrom,
                year_to: yearTo
            };

            loadMovies();
        }

        function loadMovies() {
            const stats = document.getElementById('stats');
            stats.innerHTML = 'Searching for amazing movies... ✨';

            const queryParams = new URLSearchParams(currentFilters).toString();
            fetch(`/api/movies?${queryParams}`)
                .then(response => response.json())
                .then(data => {
                    allMovies = data.movies;
                    displayMovies(allMovies);
                    stats.innerHTML = `🎉 Found ${allMovies.length} incredible movies!`;
                })
                .catch(error => {
                    stats.innerHTML = '🎬 Ready to explore movies!';
                    console.error('Error:', error);
                });
        }

        function displayMovies(movies) {
            const movieGrid = document.getElementById('movieGrid');
            movieGrid.innerHTML = '';

            if (movies.length === 0) {
                movieGrid.innerHTML = '<div class="loading">🎬 No movies found. Try different filters or search terms!</div>';
                return;
            }

            movies.forEach(movie => {
                const movieCard = document.createElement('div');
                movieCard.className = `movie-card ${currentMode === 'kids' ? 'kids-movie-card' : ''}`;

                const posterUrl = movie.poster_path ? 'https://image.tmdb.org/t/p/w500' + movie.poster_path : 'https://via.placeholder.com/300x450/2a2a2a/FFFFFF?text=No+Poster';
                const releaseYear = movie.release_date ? movie.release_date.substring(0,4) : 'Unknown';
                const languageName = movie.original_language ? getLanguageName(movie.original_language) : 'Unknown';

                movieCard.innerHTML = `
                    <img src="${posterUrl}" class="movie-poster" alt="${movie.title}" onerror="this.src='https://via.placeholder.com/300x450/2a2a2a/FFFFFF?text=No+Poster'">
                    <div class="movie-title">${movie.title}</div>
                    <div class="movie-info">
                        ⭐ ${movie.vote_average}/10 | 📅 ${releaseYear}
                    </div>
                    <div class="movie-language">🗣️ ${languageName}</div>
                    <button class="details-btn" onclick="showMovieDetails(${movie.id})">
                        🎬 View Details
                    </button>
                `;

                movieGrid.appendChild(movieCard);
            });
        }

        function displaySimilarMovies(movies) {
            const similarGrid = document.getElementById('similarGrid');
            const similarSection = document.getElementById('similarMovies');
            similarGrid.innerHTML = '';

            if (movies.length === 0) {
                similarGrid.innerHTML = '<div class="loading">No similar movies found.</div>';
                return;
            }

            movies.forEach(movie => {
                const movieCard = document.createElement('div');
                movieCard.className = `movie-card ${currentMode === 'kids' ? 'kids-movie-card' : ''}`;

                const posterUrl = movie.poster_path ? 'https://image.tmdb.org/t/p/w500' + movie.poster_path : 'https://via.placeholder.com/300x450/2a2a2a/FFFFFF?text=No+Poster';
                const releaseYear = movie.release_date ? movie.release_date.substring(0,4) : 'Unknown';
                const languageName = movie.original_language ? getLanguageName(movie.original_language) : 'Unknown';

                movieCard.innerHTML = `
                    <img src="${posterUrl}" class="movie-poster" alt="${movie.title}" onerror="this.src='https://via.placeholder.com/300x450/2a2a2a/FFFFFF?text=No+Poster'">
                    <div class="movie-title">${movie.title}</div>
                    <div class="movie-info">
                        ⭐ ${movie.vote_average}/10 | 📅 ${releaseYear}
                    </div>
                    <div class="movie-language">🗣️ ${languageName}</div>
                    <button class="details-btn" onclick="showMovieDetails(${movie.id})">
                        🎬 View Details
                    </button>
                `;

                similarGrid.appendChild(movieCard);
            });

            similarSection.style.display = 'block';
        }

        function getLanguageName(code) {
            const languages = {
                {% for code, name in languages.items() %}
                '{{ code }}': '{{ name }}',
                {% endfor %}
            };
            return languages[code] || code.toUpperCase();
        }

        function showMovieDetails(movieId) {
            fetch(`/api/movie/${movieId}`)
                .then(response => response.json())
                .then(movie => {
                    const movieGrid = document.getElementById('movieGrid');
                    const movieDetails = document.getElementById('movieDetails');
                    const similarSection = document.getElementById('similarMovies');

                    movieGrid.style.display = 'none';
                    similarSection.style.display = 'none';
                    movieDetails.style.display = 'block';

                    const trailerUrl = movie.trailer_url || '#';
                    const backdropUrl = movie.backdrop_path ? 'https://image.tmdb.org/t/p/w1280' + movie.backdrop_path : '';
                    const genres = movie.genres ? movie.genres.join(', ') : 'Unknown';
                    const cast = movie.cast ? movie.cast.slice(0, 6).join(', ') : 'No cast information';

                    movieDetails.innerHTML = `
                        ${backdropUrl ? `<img src="${backdropUrl}" style="width: 100%; border-radius: 20px; margin-bottom: 25px; max-height: 400px; object-fit: cover; border: 2px solid rgba(255,255,255,0.1);">` : ''}
                        <h2 style="color: #FFD700; margin-bottom: 10px;">${movie.title}</h2>
                        ${movie.tagline ? `<p style="font-style: italic; font-size: 1.3rem; margin: 15px 0; color: #4ECDC4;">${movie.tagline}</p>` : ''}

                        <div class="trailer-container">
                            ${movie.trailer_url ? `
                            <a href="${movie.trailer_url}" target="_blank" class="trailer-btn">
                                🎥 Watch Official Trailer
                            </a>
                            ` : '<p style="color: #FF6B6B; font-size: 1.1rem;">🎬 Trailer coming soon!</p>'}
                        </div>

                        <div style="display: grid; grid-template-columns: 2fr 1fr; gap: 40px; margin: 30px 0;">
                            <div>
                                <h3 style="color: #FFD700; margin-bottom: 15px;">📖 Story</h3>
                                <p style="line-height: 1.7; font-size: 1.1rem; color: #e0e0e0;">${movie.overview || 'No description available yet.'}</p>

                                <h3 style="color: #FFD700; margin: 25px 0 15px 0;">🎭 Star Cast</h3>
                                <p style="color: #ccc; font-size: 1rem;">${cast}</p>

                                <button class="details-btn" onclick="getSimilarMovies(${movie.id})" style="margin-top: 20px; width: auto; padding: 10px 20px;">
                                    🔍 Find Similar Movies
                                </button>
                            </div>

                            <div>
                                <h3 style="color: #FFD700; margin-bottom: 15px;">🎬 Movie Details</h3>
                                <div style="display: flex; flex-direction: column; gap: 12px;">
                                    <p><strong>Release Date:</strong> ${movie.release_date || 'Coming Soon'}</p>
                                    <p><strong>Rating:</strong> ⭐ ${movie.vote_average}/10</p>
                                    <p><strong>Duration:</strong> ${movie.runtime || 'Unknown'} minutes</p>
                                    <p><strong>Genres:</strong> ${genres}</p>
                                    <p><strong>Language:</strong> ${getLanguageName(movie.original_language)}</p>
                                    <p><strong>Budget:</strong> ${movie.budget ? '$' + movie.budget.toLocaleString() : 'Not disclosed'}</p>
                                    <p><strong>Revenue:</strong> ${movie.revenue ? '$' + movie.revenue.toLocaleString() : 'Not disclosed'}</p>
                                </div>
                            </div>
                        </div>

                        <button class="back-btn" onclick="goBack()">
                            ← Back to Movies
                        </button>
                    `;
                });
        }

        function getSimilarMovies(movieId) {
            fetch(`/api/recommendations/${movieId}`)
                .then(response => response.json())
                .then(data => {
                    if (data.recommendations && data.recommendations.length > 0) {
                        displaySimilarMovies(data.recommendations);
                        document.getElementById('stats').innerHTML = `🔍 Found ${data.recommendations.length} similar movies!`;
                    }
                });
        }

        function goBack() {
            document.getElementById('movieGrid').style.display = 'grid';
            document.getElementById('movieDetails').style.display = 'none';
            document.getElementById('similarMovies').style.display = 'none';
        }

        // Initialize when page loads
        window.onload = function() {
            initializeFilters();
            applyFilters();
        }

        // Search as you type
        let searchTimeout;
        document.getElementById('searchInput').addEventListener('input', function() {
            clearTimeout(searchTimeout);
            searchTimeout = setTimeout(applyFilters, 500);
        });
    </script>
</body>
</html>
"""

In [20]:
# Cell 6: Simplified Flask Routes (No AI, Only Similar Movies)
@app.route('/')
def home():
    genres_list = [{'id': k, 'name': v} for k, v in GENRES.items()]
    return render_template_string(HTML_TEMPLATE,
                                genres=genres_list,
                                countries=COUNTRIES,
                                languages=LANGUAGES,
                                indian_languages=INDIAN_LANGUAGES,
                                kids_animations=KIDS_BACKGROUND_ANIMATIONS)

@app.route('/api/movies')
def api_movies():
    mode = request.args.get('mode', 'regular')
    search_query = request.args.get('search', '')
    genre = request.args.get('genre', '')
    language = request.args.get('language', '')
    year_from = request.args.get('year_from', '1980')
    year_to = request.args.get('year_to', '2025')

    try:
        if search_query:
            movies = search_movies_enhanced(search_query)
        elif mode == 'kids':
            movies = fetch_kids_movies_enhanced()
        else:
            movies = fetch_movies_with_filters(
                genre=genre,
                language=language,
                year_from=int(year_from),
                year_to=int(year_to)
            )
            # Add recent movies and popular movies if no specific filters
            if not genre and not language and not search_query:
                recent_movies = fetch_recent_movies()
                popular_movies = fetch_popular_movies(pages=3)
                movies.extend(recent_movies)
                movies.extend(popular_movies)

        # Remove duplicates
        seen_ids = set()
        unique_movies = []
        for movie in movies:
            if movie['id'] not in seen_ids:
                seen_ids.add(movie['id'])
                unique_movies.append(movie)

        return jsonify({'movies': unique_movies[:100]})

    except Exception as e:
        print(f"API Error: {e}")
        return jsonify({'error': str(e), 'movies': []})

@app.route('/api/movie/<int:movie_id>')
def api_movie(movie_id):
    try:
        movie_data = fetch_movie_details_enhanced(movie_id)
        if movie_data:
            trailer_url = get_youtube_trailer(movie_data)
            cast = [member['name'] for member in movie_data.get('credits', {}).get('cast', [])[:8]]
            genres = [genre['name'] for genre in movie_data.get('genres', [])]

            return jsonify({
                'id': movie_data.get('id'),
                'title': movie_data.get('title'),
                'overview': movie_data.get('overview'),
                'poster_path': movie_data.get('poster_path'),
                'backdrop_path': movie_data.get('backdrop_path'),
                'release_date': movie_data.get('release_date'),
                'vote_average': movie_data.get('vote_average'),
                'runtime': movie_data.get('runtime'),
                'tagline': movie_data.get('tagline'),
                'budget': movie_data.get('budget'),
                'revenue': movie_data.get('revenue'),
                'original_language': movie_data.get('original_language'),
                'trailer_url': trailer_url,
                'cast': cast,
                'genres': genres
            })
        return jsonify({'error': 'Movie not found'}), 404
    except Exception as e:
        return jsonify({'error': str(e)}), 500

@app.route('/api/recommendations/<int:movie_id>')
def api_recommendations(movie_id):
    try:
        # Get a good mix of movies for recommendations
        popular_movies = fetch_popular_movies(pages=2)
        recent_movies = fetch_recent_movies()
        all_movies = popular_movies + recent_movies

        # Remove duplicates
        seen_ids = set()
        unique_movies = []
        for movie in all_movies:
            if movie['id'] not in seen_ids:
                seen_ids.add(movie['id'])
                unique_movies.append(movie)

        # Find similar movies based on genres
        target_movie = None
        for movie in unique_movies:
            if movie['id'] == movie_id:
                target_movie = movie
                break

        recommendations = []
        if target_movie:
            target_genres = set(target_movie.get('genre_ids', []))
            for movie in unique_movies:
                if movie['id'] != movie_id:
                    movie_genres = set(movie.get('genre_ids', []))
                    similarity = len(target_genres.intersection(movie_genres))
                    if similarity > 0:
                        recommendations.append((movie, similarity))

            # Sort by similarity score
            recommendations.sort(key=lambda x: x[1], reverse=True)
            recommendations = [movie for movie, score in recommendations[:12]]

        print(f"Found {len(recommendations)} similar movies for {movie_id}")
        return jsonify({
            'recommendations': recommendations[:12],
            'count': len(recommendations)
        })

    except Exception as e:
        print(f"Error in recommendations: {e}")
        # Return some popular movies as fallback
        popular_movies = fetch_popular_movies(pages=1)
        return jsonify({
            'recommendations': popular_movies[:8],
            'error': str(e)
        })

print("✅ CLEAN MOVIE SYSTEM READY!")
print("🎯 Features:")
print("   • 🔍 Only Similar Movies (No AI)")
print("   • 🎬 Clean Interface")
print("   • 🎪 Kids Mode Clears Properly")
print("   • 📱 Similar Movies Show in Same Page")

✅ CLEAN MOVIE SYSTEM READY!
🎯 Features:
   • 🔍 Only Similar Movies (No AI)
   • 🎬 Clean Interface
   • 🎪 Kids Mode Clears Properly
   • 📱 Similar Movies Show in Same Page


In [21]:
# Cell 7: Start the Enhanced Server
from google.colab.output import eval_js
import threading
import time

def run_flask():
    app.run(host='0.0.0.0', port=5000, debug=False, threaded=True)

# Start Flask in background thread
thread = threading.Thread(target=run_flask, daemon=True)
thread.start()

# Wait for server to start
time.sleep(3)

# Get the public URL
public_url = eval_js("google.colab.kernel.proxyPort(5000)")
print(f"""
🎬 🌟 🚀 CINEMATIC MASTERPIECE READY! 🚀 🌟 🎬

🌐 YOUR ULTIMATE MOVIE DISCOVERY PLATFORM:
🔗 {public_url}

✨ FEATURE HIGHLIGHTS:
   • 🎨 STUNNING VISUAL DESIGN with Glass Morphism
   • 🤖 ADVANCED AI with Multi-Factor Analysis
   • 🎯 SMART RECOMMENDATIONS that learn your taste
   • 🌈 BEAUTIFUL ANIMATIONS & Sparkle Effects
   • 🎪 INTERACTIVE KIDS MODE with Fun Animations
   • 🔥 TRENDING MOVIE Highlights
   • 📊 AI INSIGHTS Dashboard
   • 🚀 LIGHTNING FAST Performance

🎯 UNIQUE SELLING POINTS:
   • Content-Based Filtering + Collaborative Features
   • Personalized Movie Discovery
   • Family-Friendly Safe Mode
   • Advanced Search & Filters
   • YouTube Trailers Integration
   • Responsive Mobile Design

🌟 GET READY FOR THE ULTIMATE MOVIE EXPERIENCE!
""")

 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.



🎬 🌟 🚀 CINEMATIC MASTERPIECE READY! 🚀 🌟 🎬

🌐 YOUR ULTIMATE MOVIE DISCOVERY PLATFORM:
🔗 https://5000-m-s-38n5e52r5180w-c.us-east1-1.prod.colab.dev

✨ FEATURE HIGHLIGHTS:
   • 🎨 STUNNING VISUAL DESIGN with Glass Morphism
   • 🤖 ADVANCED AI with Multi-Factor Analysis
   • 🎯 SMART RECOMMENDATIONS that learn your taste
   • 🌈 BEAUTIFUL ANIMATIONS & Sparkle Effects
   • 🎪 INTERACTIVE KIDS MODE with Fun Animations
   • 🔥 TRENDING MOVIE Highlights
   • 📊 AI INSIGHTS Dashboard
   • 🚀 LIGHTNING FAST Performance

🎯 UNIQUE SELLING POINTS:
   • Content-Based Filtering + Collaborative Features
   • Personalized Movie Discovery
   • Family-Friendly Safe Mode
   • Advanced Search & Filters
   • YouTube Trailers Integration
   • Responsive Mobile Design

🌟 GET READY FOR THE ULTIMATE MOVIE EXPERIENCE!

